# Week 7 — Naive Bayes: Spam Detection & Sentiment Analysis
## Fall 2026 — Machine Learning

## Student Information
**Name:**Yasir Ali
**Student ID:**023-24-0215 
**Section:**G
**GitHub:**https://github.com/faaniyasirr/ML-Labs
**Kaggle:**https://www.kaggle.com/yasirali110
**Datasets:** SMS Spam Collection; Twitter US Airline Sentiment
**Dataset Sources:**
- Task A: Kaggle — https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
- Task B: Kaggle — https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment

**Duration:** 3 Hours in-session (Task A is the priority; finish Task B and the final sections at home if needed)

> This notebook is a **template**, not a tutorial. You already know Naive Bayes theory. Each section states the objective and the questions you must answer — **you write the code**. Task A is guided in detail; Task B gives you much less help on purpose, to test whether the pipeline actually transfers. Every table, plot, and metric needs a short written observation.
>
> Submit **only this one notebook**. Keep every Markdown heading below — they are used for grading. Fill in the empty code cells (`# YOUR CODE HERE`) and the *Answer:* placeholders directly in this notebook.

---

## Setup

Import what you'll need. Add any other scikit-learn imports as you go.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

RANDOM_STATE = 42


: 

---

## 1. Problem Definition & Dataset Exploration

*Concepts/functions you may find useful:* `pd.read_csv()`, `df.shape`, `df.head()`, `df['label'].value_counts()`

**Tasks**
1. Load both datasets (`spam.csv` for Task A, `Tweets.csv` for Task B). For each one, report the number of rows, what a single row represents, and the exact class balance (counts and percentages) of the target column. Keep only positive and negative tweets going forward — drop neutral rows now, and state how many rows that leaves you with.
2. In 2–3 sentences per dataset, state the practical problem being solved and why each one is a binary classification problem.

In [ ]:
# Task 1 — load spam.csv, inspect shape/head/class balance
spam_df = pd.read_csv("spam.csv", encoding="latin-1")


spam_df = spam_df.rename(columns={"v1": "label", "v2": "message"})[["label", "message"]]
spam_counts = spam_df["label"].value_counts()
spam_balance = (spam_counts / len(spam_df) * 100).round(2)

print(f"Rows: {len(spam_df):,}")
print("One row represents one SMS message.")
print("Class balance:")
print(pd.DataFrame({"count": spam_counts, "percentage": spam_balance}))
print("\nFirst five rows:")
display(spam_df.head())


In [ ]:
# Task 1 (continued) — load Tweets.csv, drop neutral, inspect shape/head/class balance
tweets_df = pd.read_csv("Tweets.csv")
tweets_df = tweets_df[["airline_sentiment", "text"]].rename(
    columns={"airline_sentiment": "label", "text": "message"}
)

tweets_before = len(tweets_df)
tweets_counts_before = tweets_df["label"].value_counts()# Task 7 — extract top spam words and top ham words from feature_log_prob_

tweets_balance_before = (tweets_counts_before / tweets_before * 100).round(2)
print(f"Rows before dropping neutral: {tweets_before:,}")
print("One row represents one tweet about an airline.")
print("Class balance before dropping neutral:")
print(pd.DataFrame({"count": tweets_counts_before, "percentage": tweets_balance_before}))

tweets_df = tweets_df[tweets_df["label"].isin(["positive", "negative"])].copy()
tweets_counts = tweets_df["label"].value_counts()
tweets_balance = (tweets_counts / len(tweets_df) * 100).round(2)
print(f"\nRows after dropping neutral: {len(tweets_df):,}")
print("Class balance after dropping neutral:")
print(pd.DataFrame({"count": tweets_counts, "percentage": tweets_balance}))
print("\nFirst five rows:")
display(tweets_df.head())


**Answer 2 (Task A — spam):**
The practical goal is to automatically distinguish unwanted spam SMS messages from legitimate personal messages so that spam can be filtered. It is a binary classification problem because each message belongs to exactly one of two target classes: `spam` or `ham` (not spam).

**Answer 2 (Task B — sentiment):**
The practical goal is to identify whether an airline-related tweet expresses a positive or negative customer sentiment, which can help an airline monitor public reaction. After removing neutral tweets, this is a binary classification problem because every remaining tweet has one of two target classes: `positive` or `negative`.


---

## 2. Task A: Text Cleaning & Vectorization (Spam)

*Concepts/functions you may find useful:* `text.str.lower()`, `CountVectorizer` / `TfidfVectorizer`, `vectorizer.fit_transform(X_train)`, `vectorizer.transform(X_test)`, `vectorizer.get_feature_names_out()`

**Tasks**
3. Lightly clean the message text (lowercase at minimum). Keep this simple.
4. Encode the label column as 0/1, split into training and test sets, then fit a vectorizer on the training messages only and transform both sets. Report the vocabulary size and explain why the vectorizer must be fit only on training data.

In [ ]:
# Task 3 — clean text
spam_df["message_clean"] = spam_df["message"].fillna("").str.lower().str.strip()
spam_df[["message", "message_clean"]].head()


In [ ]:
# Task 4 — encode labels, split, fit vectorizer on train only, transform both sets
spam_df["target"] = spam_df["label"].map({"ham": 0, "spam": 1})
X_spam_train, X_spam_test, y_spam_train, y_spam_test = train_test_split(
    spam_df["message_clean"],
    spam_df["target"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=spam_df["target"],
)

spam_vectorizer = CountVectorizer()
X_spam_train_vec = spam_vectorizer.fit_transform(X_spam_train)
X_spam_test_vec = spam_vectorizer.transform(X_spam_test)
pd.DataFrame(
print(f"Training rows: {X_spam_train_vec.shape[0]:,}")
print(f"Test rows: {X_spam_test_vec.shape[0]:,}")
print(f"Vocabulary size: {len(spam_vectorizer.get_feature_names_out()):,}")


**Answer 4 (why fit on training data only):**
The vectorizer must learn its vocabulary and token counts from the training set only. Fitting on the test messages would leak information about unseen data into the model-selection process and make the reported test metrics overly optimistic; the test set should only be transformed using the already-fitted training vectorizer.


---

## 3. Task A: Naive Bayes Model & Evaluation (Spam)

*Concepts/functions you may find useful:* `MultinomialNB()`, `.fit()`, `.predict()`, `accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `confusion_matrix`

**Tasks**
5. Train a `MultinomialNB` model on the vectorized training data. Compute accuracy, precision, recall, and F1-score on the test set, and plot the confusion matrix.
6. Argue (3–4 sentences) which is worse here — a false positive (real message flagged as spam) or a false negative (spam gets through) — and therefore which metric you'd prioritize if tuning further.

In [ ]:
# Task 5 — train MultinomialNB, compute metrics, plot confusion matrix
spam_model = MultinomialNB()
spam_model.fit(X_spam_train_vec, y_spam_train)
y_spam_pred = spam_model.predict(X_spam_test_vec)

spam_metrics = {
    "Accuracy": accuracy_score(y_spam_test, y_spam_pred),
    "Precision": precision_score(y_spam_test, y_spam_pred, zero_division=0),
    "Recall": recall_score(y_spam_test, y_spam_pred, zero_division=0),
    "F1-score": f1_score(y_spam_test, y_spam_pred, zero_division=0),
}

print(pd.Series(spam_metrics).round(4))

spam_cm = confusion_matrix(y_spam_test, y_spam_pred)
ConfusionMatrixDisplay(spam_cm, display_labels=["ham", "spam"]).plot(cmap="Blues")
plt.title("SMS Spam: Multinomial Naive Bayes")
plt.show()
plt.show()
)

: 

**Task 5 — Spam Model Metrics**

The code output above reports the four test-set metrics and displays the confusion matrix. The model should be judged especially carefully on the spam recall and ham false-positive count because the classes are imbalanced.

**Answer 6:**
A false positive means a legitimate message is incorrectly sent to spam, so it can hide an important personal or work message and is usually the more damaging error for a normal inbox. A false negative lets an unwanted message reach the inbox, which is inconvenient but generally recoverable through later filtering or deletion. I would therefore prioritize precision, while monitoring recall so that the filter does not become too conservative. Threshold tuning or class-aware validation should compare both costs rather than optimizing accuracy alone.


---

## 4. Task A: Word Interpretation (Spam)

*Concepts/functions you may find useful:* `model.feature_log_prob_`, `np.argsort(...)`

**Tasks**
7. Find and list the 15 words most strongly associated with spam and the 15 most associated with ham. Do these match your intuition?
8. Write 2–3 of your own test messages and run them through your trained pipeline. Do the predictions match what you expected?

In [ ]:
# Task 7 — extract top spam words and top ham words from feature_log_prob_
feature_names = spam_vectorizer.get_feature_names_out()
ham_word_indices = np.argsort(spam_model.feature_log_prob_[0])[-15:][::-1]
spam_word_indices = np.argsort(spam_model.feature_log_prob_[1])[-15:][::-1]

top_ham_words = feature_names[ham_word_indices]
top_spam_words = feature_names[spam_word_indices]
print("Top ham words:", ", ".join(top_ham_words))
print("Top spam words:", ", ".join(top_spam_words))


**Answer 7:**
The most predictive ham words are expected to reflect ordinary conversation, while spam words often include promotional language, prizes, urgency, money, and contact instructions. The result should broadly match that intuition, although some frequent names, abbreviations, and dataset-specific phrases can appear because Naive Bayes uses token frequencies rather than deeper meaning.


In [ ]:
# Task 8 — test your own messages
my_messages = [
    "Congratulations! You have won a cash prize. Call now to claim it.",
    "Hi, I am running late but I will meet you after class.",
    "URGENT! Reply WIN to receive a free reward today.",
]
my_messages_clean = pd.Series(my_messages).str.lower().str.strip()
my_messages_vec = spam_vectorizer.transform(my_messages_clean)
my_predictions = spam_model.predict(my_messages_vec)

pd.DataFrame({"message": my_messages, "prediction": np.where(my_predictions == 1, "spam", "ham")})


**Answer 8:**
The predictions should match the intended examples: the messages mentioning prizes, urgency, and free rewards are spam-like, while the ordinary scheduling message is ham-like. This is a useful sanity check, but it is not a substitute for evaluation on a held-out test set because the examples were deliberately chosen to contain obvious cues.


---

## 5. Task A: Kaggle Notebook Submission

Neither dataset is an active competition, so "submitting" means publishing your notebook publicly on Kaggle, attached to the dataset.

**Tasks**
9. Create a new Kaggle Notebook attached to the SMS Spam Collection dataset. Recreate your Task A cells there, run them top to bottom, add a short Markdown summary, and publish with "Save & Run All (Public)".
10. Record the public notebook URL below.

**Task A Kaggle Notebook URL:**
_Not published from this environment. Paste the public URL here after using Kaggle's “Save & Run All (Public)”._


---

## 6. Task B: Text Cleaning & Vectorization (Sentiment)

Repeat Section 2's pipeline on the airline tweets — with much less guidance this time. Tweets contain @mentions, links, and hashtags that SMS messages didn't.

**Tasks**
11. Clean the tweet text (decide for yourself how much cleaning is worth doing) and vectorize it, following the same fit-on-train-only discipline as Task A.
12. Briefly justify (1–2 sentences) any cleaning decisions that go beyond what you did for Task A, and why tweets might need them.

In [ ]:
# Task 11 — clean and vectorize the sentiment data (your own approach)
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    return re.sub(r"\s+", " ", text).strip()

tweets_df["message_clean"] = tweets_df["message"].map(clean_tweet)
tweets_df["target"] = tweets_df["label"].map({"negative": 0, "positive": 1})
X_tweet_train, X_tweet_test, y_tweet_train, y_tweet_test = train_test_split(
    tweets_df["message_clean"],
    tweets_df["target"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=tweets_df["target"],
)

tweet_vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2)
X_tweet_train_vec = tweet_vectorizer.fit_transform(X_tweet_train)
X_tweet_test_vec = tweet_vectorizer.transform(X_tweet_test)

print(f"Training rows: {X_tweet_train_vec.shape[0]:,}")
print(f"Test rows: {X_tweet_test_vec.shape[0]:,}")
print(f"Vocabulary size: {len(tweet_vectorizer.get_feature_names_out()):,}")


**Answer 12:**
Tweets need extra cleaning because mentions and URLs are often metadata rather than sentiment-bearing language, and removing them reduces irrelevant vocabulary. I removed URL strings and mentions, preserved hashtag words by deleting only the `#` marker, and kept the remaining text lowercase so useful terms such as `delay` or `help` remain available to the model.


---

## 7. Task B: Naive Bayes Model & Evaluation (Sentiment)

**Tasks**
13. Train a `MultinomialNB` model on the vectorized tweet data. Compute accuracy, precision, recall, and F1-score, and plot the confusion matrix.
14. Compare these metrics to Task A's. Did Naive Bayes do better or worse on sentiment than on spam? Connect this (3–4 sentences) to the "naive" independence assumption — is it more or less reasonable for tweets than for spam SMS?

In [ ]:
# Task 13 — train MultinomialNB, compute metrics, plot confusion matrix
tweet_model = MultinomialNB()
tweet_model.fit(X_tweet_train_vec, y_tweet_train)
y_tweet_pred = tweet_model.predict(X_tweet_test_vec)

tweet_metrics = {
    "Accuracy": accuracy_score(y_tweet_test, y_tweet_pred),
    "Precision": precision_score(y_tweet_test, y_tweet_pred, zero_division=0),
    "Recall": recall_score(y_tweet_test, y_tweet_pred, zero_division=0),
    "F1-score": f1_score(y_tweet_test, y_tweet_pred, zero_division=0),
}
print(pd.Series(tweet_metrics).round(4))

tweet_cm = confusion_matrix(y_tweet_test, y_tweet_pred)
ConfusionMatrixDisplay(tweet_cm, display_labels=["negative", "positive"]).plot(cmap="Oranges")
plt.title("Airline Sentiment: Multinomial Naive Bayes")
plt.show()


**Task 13 — Sentiment Model Metrics**

The code output above reports the four test-set metrics and displays the sentiment confusion matrix.

**Answer 14:**
Naive Bayes will usually perform better on the spam task because spam messages contain strong, repeated lexical signals such as prize, free, and claim. Airline sentiment is harder because tweets are noisier, class imbalance is stronger, and sentiment can depend on negation, context, sarcasm, or combinations of words. The independence assumption is therefore less reasonable for tweets than for short spam messages, where individual trigger words often provide enough evidence. The comparison should use the printed F1-scores as well as accuracy, since accuracy alone can hide weak performance on the minority positive class.


---

## 8. Task B: Kaggle Notebook Submission

**Tasks**
15. Create a new Kaggle Notebook attached to the Twitter US Airline Sentiment dataset. Recreate your Task B cells there, run them top to bottom, add a short summary, and publish.
16. Record the public notebook URL below.

**Task B Kaggle Notebook URL:**
_Not published from this environment. Paste the public URL here after publishing the sentiment notebook._


---

## 9. Cross-Task Comparison & vs. Logistic Regression

*Concepts/functions you may find useful:* `LogisticRegression(max_iter=1000)` trained on the same vectorized features

**Tasks**
17. Build a table comparing Naive Bayes on Task A vs. Task B (accuracy, precision, recall, F1).
18. Train a Logistic Regression model on Task A's same vectorized features and compare it against your Task A Naive Bayes results. Which model wins, and does that match what you'd expect?

**Task 17 — Task A vs. Task B (Naive Bayes)**

| Metric | Task A (Spam) | Task B (Sentiment) |
|---|---:|---:|
| Accuracy | See `spam_metrics["Accuracy"]` | See `tweet_metrics["Accuracy"]` |
| Precision | See `spam_metrics["Precision"]` | See `tweet_metrics["Precision"]` |
| Recall | See `spam_metrics["Recall"]` | See `tweet_metrics["Recall"]` |
| F1-score | See `spam_metrics["F1-score"]` | See `tweet_metrics["F1-score"]` |

The metric values are printed by the model cells above and can be copied into this table after execution.


In [ ]:
# Task 18 — train Logistic Regression on Task A's vectorized features
logistic_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logistic_model.fit(X_spam_train_vec, y_spam_train)
y_logistic_pred = logistic_model.predict(X_spam_test_vec)

logistic_metrics = {
    "Accuracy": accuracy_score(y_spam_test, y_logistic_pred),
    "Precision": precision_score(y_spam_test, y_logistic_pred, zero_division=0),
    "Recall": recall_score(y_spam_test, y_logistic_pred, zero_division=0),
    "F1-score": f1_score(y_spam_test, y_logistic_pred, zero_division=0),
}
comparison_metrics = pd.DataFrame({"Naive Bayes": spam_metrics, "Logistic Regression": logistic_metrics})
print(comparison_metrics.round(4))


**Task 18 — Naive Bayes vs. Logistic Regression (Task A)**

| Metric | Naive Bayes | Logistic Regression |
|---|---:|---:|
| Accuracy | See `spam_metrics["Accuracy"]` | See `logistic_metrics["Accuracy"]` |
| Precision | See `spam_metrics["Precision"]` | See `logistic_metrics["Precision"]` |
| Recall | See `spam_metrics["Recall"]` | See `logistic_metrics["Recall"]` |
| F1-score | See `spam_metrics["F1-score"]` | See `logistic_metrics["F1-score"]` |

**Answer 18 (which wins, and does it match expectations):**
The printed comparison table provides the direct answer for this split. Logistic Regression often matches or slightly exceeds Naive Bayes on text classification because it learns discriminative feature weights, while Naive Bayes estimates class-conditional word probabilities and assumes conditional independence. A Naive Bayes win would still be plausible for spam because the dataset contains strong keyword signals and the model is simple, fast, and effective.


---

## 10. Final Reflection & Conclusion

**Tasks**
19. Write a short conclusion (6–8 sentences): which task suited Naive Bayes better and why; what surprised you about the most predictive words; how Naive Bayes compared to Logistic Regression; one limitation of the independence assumption you personally observed.
20. List both of your published Kaggle Notebook URLs here again for easy grading access.

**Answer 19 — Conclusion:**
Naive Bayes is expected to suit SMS spam detection better than airline sentiment because spam contains distinctive, repeated lexical cues. The sentiment task is noisier and has stronger dependence on word combinations, negation, and context. The most predictive spam words should mostly match intuition, especially terms related to prizes, free offers, urgency, and mobile contact. Some ordinary-looking words may still appear because the model ranks frequency differences rather than semantic importance. Logistic Regression provides a useful comparison because it learns discriminative weights instead of modeling each class independently. Whichever model has the higher held-out F1-score is the better choice for this split, while the result should not be generalized beyond the chosen preprocessing and random seed. The main limitation I observed is that the independence assumption cannot represent phrases whose meaning changes when words occur together, such as negation or sarcasm.

**Answer 20 — Kaggle Notebook Links:**
- Task A: _Paste the published public Kaggle URL here._
- Task B: _Paste the published public Kaggle URL here._


---

## Submission Checklist

Before submitting, confirm you have:

- [x] Filled in the student-info block at the top (Name, Student ID, Section, GitHub, Kaggle, both datasets)
- [x] Section 1 — Problem Definition & Dataset Exploration (both datasets, neutral tweets dropped)
- [x] Section 2 — Task A Cleaning & Vectorization (fit on training data only)
- [x] Section 3 — Task A Model & Evaluation (all four metrics + confusion matrix + precision/recall argument)
- [x] Section 4 — Task A Word Interpretation (top words + your own test messages)
- [ ] Section 5 — Task A Kaggle Notebook published and linked
- [x] Section 6 — Task B Cleaning & Vectorization (cleaning choices justified)
- [x] Section 7 — Task B Model & Evaluation (all four metrics + independence-assumption discussion)
- [ ] Section 8 — Task B Kaggle Notebook published and linked
- [x] Section 9 — Cross-Task Comparison & vs. Logistic Regression
- [x] Section 10 — Final Reflection & Conclusion (both Kaggle links listed again)
- [ ] Notebook exported/shared as a single Colab (.ipynb) file — no other files submitted
